In [2]:
"""
NONLINEAR SUPPLY CHAIN (Pyomo) — FULL WORKING SCRIPT
====================================================

Why your previous run failed:
- Your objective contains x^2 terms (quadratic / nonlinear).
- If Pyomo tries to write the model in LP format (GLPK/CBC/HiGHS-LP),
  it throws: "contains nonlinear terms that cannot be written to LP format".

This script solves it correctly by using:
- IPOPT (recommended for continuous nonlinear problems), OR
- GUROBI/CPLEX (recommended if you want QP/QCQP support).

IMPORTANT:
- This model is CONTINUOUS (no binary/integer variables).
- IPOPT cannot solve integer variables (MINLP). For MINLP you'd need other solvers.

---------------------------------------------------------------------------
WHAT THE MODEL DOES (Supply Chain):
- Plants produce product each period
- Plants ship to DCs
- DCs hold inventory and ship to customers
- Customer demand must be met each period
- Objective has nonlinear (quadratic) cost terms to model congestion/overtime
---------------------------------------------------------------------------

Author style: super-commented so you can reuse as a template.
"""

from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals,
    Constraint, Objective, minimize, value, SolverFactory
)


# =============================================================================
# 1) BUILD MODEL
# =============================================================================
def build_model():
    m = ConcreteModel(name="Nonlinear_SC_QP_QCQP")

    # -------------------------------------------------------------------------
    # SETS
    # -------------------------------------------------------------------------
    m.P = Set(initialize=["PL1", "PL2"])          # Plants
    m.D = Set(initialize=["DC1", "DC2"])          # Distribution Centers
    m.C = Set(initialize=["C1", "C2", "C3"])      # Customers
    m.T = Set(initialize=[1, 2, 3, 4])            # Time periods

    # -------------------------------------------------------------------------
    # PARAMETERS
    # -------------------------------------------------------------------------
    # Demand[c,t]
    demand_data = {
        ("C1", 1): 40, ("C1", 2): 35, ("C1", 3): 50, ("C1", 4): 45,
        ("C2", 1): 30, ("C2", 2): 45, ("C2", 3): 35, ("C2", 4): 40,
        ("C3", 1): 25, ("C3", 2): 20, ("C3", 3): 30, ("C3", 4): 25,
    }
    m.Demand = Param(m.C, m.T, initialize=demand_data, within=NonNegativeReals)

    # Capacity at plant p in time t
    cap_data = {
        ("PL1", 1): 120, ("PL1", 2): 120, ("PL1", 3): 110, ("PL1", 4): 120,
        ("PL2", 1):  90, ("PL2", 2):  95, ("PL2", 3):  90, ("PL2", 4):  95,
    }
    m.Cap = Param(m.P, m.T, initialize=cap_data, within=NonNegativeReals)

    # Production cost = a[p]*Prod + b[p]*Prod^2  (quadratic = nonlinear)
    m.aProd = Param(m.P, initialize={"PL1": 2.0, "PL2": 1.8})
    m.bProd = Param(m.P, initialize={"PL1": 0.010, "PL2": 0.015})  # keep >=0 for convexity

    # Inventory holding at DC
    m.Hold = Param(m.D, initialize={"DC1": 0.20, "DC2": 0.25})

    # Shipping costs (linear + quadratic)
    m.cPD = Param(m.P, m.D, initialize={
        ("PL1", "DC1"): 0.8, ("PL1", "DC2"): 1.1,
        ("PL2", "DC1"): 1.0, ("PL2", "DC2"): 0.7,
    })
    m.cDC = Param(m.D, m.C, initialize={
        ("DC1", "C1"): 0.6, ("DC1", "C2"): 0.9, ("DC1", "C3"): 1.2,
        ("DC2", "C1"): 1.0, ("DC2", "C2"): 0.7, ("DC2", "C3"): 0.8,
    })

    # Quadratic shipping coefficients (>=0 for convexity)
    m.qPD = Param(m.P, m.D, initialize={
        ("PL1", "DC1"): 0.002,  ("PL1", "DC2"): 0.003,
        ("PL2", "DC1"): 0.002,  ("PL2", "DC2"): 0.0025,
    })
    m.qDC = Param(m.D, m.C, initialize={
        ("DC1", "C1"): 0.0015, ("DC1", "C2"): 0.0018, ("DC1", "C3"): 0.0022,
        ("DC2", "C1"): 0.0017, ("DC2", "C2"): 0.0014, ("DC2", "C3"): 0.0016,
    })

    # Initial inventory
    m.InitInv = Param(m.D, initialize={"DC1": 20, "DC2": 10})

    # OPTIONAL quadratic "stress/emissions" budget
    # If you want to disable it: set huge value like 1e12 before solving.
    m.QuadBudget = Param(initialize=1.0e6)

    # -------------------------------------------------------------------------
    # DECISION VARIABLES
    # -------------------------------------------------------------------------
    m.Prod   = Var(m.P, m.T, domain=NonNegativeReals)          # production
    m.ShipPD = Var(m.P, m.D, m.T, domain=NonNegativeReals)     # plant -> dc
    m.ShipDC = Var(m.D, m.C, m.T, domain=NonNegativeReals)     # dc -> customer
    m.Inv    = Var(m.D, m.T, domain=NonNegativeReals)          # ending inventory at DC

    # -------------------------------------------------------------------------
    # CONSTRAINTS
    # -------------------------------------------------------------------------

    # (1) Plant capacity
    def cap_rule(m, p, t):
        return m.Prod[p, t] <= m.Cap[p, t]
    m.CapConstraint = Constraint(m.P, m.T, rule=cap_rule)

    # (2) Plant balance: production must equal total outbound to DCs
    def plant_balance_rule(m, p, t):
        return m.Prod[p, t] == sum(m.ShipPD[p, d, t] for d in m.D)
    m.PlantBalance = Constraint(m.P, m.T, rule=plant_balance_rule)

    # (3) DC inventory balance:
    # Inv[d,t] = Inv[d,t-1] + inbound - outbound
    def dc_inventory_rule(m, d, t):
        inbound  = sum(m.ShipPD[p, d, t] for p in m.P)
        outbound = sum(m.ShipDC[d, c, t] for c in m.C)

        first_t = min(m.T)
        if t == first_t:
            return m.Inv[d, t] == m.InitInv[d] + inbound - outbound
        else:
            return m.Inv[d, t] == m.Inv[d, t-1] + inbound - outbound
    m.DCInventory = Constraint(m.D, m.T, rule=dc_inventory_rule)

    # (4) Demand satisfaction: deliveries >= demand each period
    def demand_rule(m, c, t):
        return sum(m.ShipDC[d, c, t] for d in m.D) >= m.Demand[c, t]
    m.DemandConstraint = Constraint(m.C, m.T, rule=demand_rule)

    # (5) OPTIONAL quadratic budget (QCQP-type constraint):
    # sum(q * flow^2) <= budget
    def quad_budget_rule(m):
        total_quad = 0.0

        # plant -> dc arcs
        for p in m.P:
            for d in m.D:
                for t in m.T:
                    total_quad += m.qPD[p, d] * (m.ShipPD[p, d, t] ** 2)

        # dc -> customer arcs
        for d in m.D:
            for c in m.C:
                for t in m.T:
                    total_quad += m.qDC[d, c] * (m.ShipDC[d, c, t] ** 2)

        return total_quad <= m.QuadBudget
    m.QuadraticBudget = Constraint(rule=quad_budget_rule)

    # -------------------------------------------------------------------------
    # OBJECTIVE (NONLINEAR because of squares)
    # -------------------------------------------------------------------------
    def objective_rule(m):
        # Production cost (linear + quadratic)
        prod_cost = sum(
            m.aProd[p] * m.Prod[p, t] + m.bProd[p] * (m.Prod[p, t] ** 2)
            for p in m.P for t in m.T
        )

        # Shipping plant->dc cost (linear + quadratic)
        ship_pd_cost = sum(
            m.cPD[p, d] * m.ShipPD[p, d, t] + m.qPD[p, d] * (m.ShipPD[p, d, t] ** 2)
            for p in m.P for d in m.D for t in m.T
        )

        # Shipping dc->customer cost (linear + quadratic)
        ship_dc_cost = sum(
            m.cDC[d, c] * m.ShipDC[d, c, t] + m.qDC[d, c] * (m.ShipDC[d, c, t] ** 2)
            for d in m.D for c in m.C for t in m.T
        )

        # Inventory holding
        hold_cost = sum(m.Hold[d] * m.Inv[d, t] for d in m.D for t in m.T)

        return prod_cost + ship_pd_cost + ship_dc_cost + hold_cost

    m.Obj = Objective(rule=objective_rule, sense=minimize)

    return m


# =============================================================================
# 2) SOLVE MODEL (correctly, without LP writer issues)
# =============================================================================
def solve_model(m, preferred_solver="ipopt"):
    """
    preferred_solver options:
    - "ipopt" (default): best for continuous nonlinear (NLP)
    - "gurobi" or "cplex": best for QP/QCQP if you have licenses
    - "gurobi_direct": sometimes avoids writer/interface issues

    DO NOT use:
    - glpk, cbc, highs (LP/MILP solvers)  -> they cannot handle x^2 in objective
    """

    opt = SolverFactory(preferred_solver)

    if not opt.available(False):
        # fallback order (still avoiding LP-only solvers)
        for s in ["gurobi_direct", "gurobi", "cplex", "ipopt"]:
            opt2 = SolverFactory(s)
            if opt2.available(False):
                opt = opt2
                preferred_solver = s
                break
        else:
            raise RuntimeError(
                "No suitable nonlinear/quadratic solver found.\n"
                "Install IPOPT (free) or use Gurobi/CPLEX.\n"
                "LP solvers (glpk/cbc/highs) will fail with x^2 terms."
            )

    results = opt.solve(m, tee=True)
    return preferred_solver, results


# =============================================================================
# 3) PRINT RESULTS (human readable)
# =============================================================================
def print_solution(m, solver_name):
    print("\n" + "=" * 70)
    print(f"SOLVED WITH: {solver_name}")
    print(f"Objective value: {value(m.Obj):.4f}")
    print("=" * 70 + "\n")

    print("PRODUCTION (per plant, per period):")
    for t in sorted(m.T):
        for p in m.P:
            print(f"  t={t:>2}  {p}: Prod = {value(m.Prod[p, t]):>8.3f}")
    print()

    print("ENDING INVENTORY (per DC, per period):")
    for t in sorted(m.T):
        for d in m.D:
            print(f"  t={t:>2}  {d}: Inv  = {value(m.Inv[d, t]):>8.3f}")
    print()

    print("DELIVERIES (to customers, per period):")
    for t in sorted(m.T):
        for c in m.C:
            delivered = sum(value(m.ShipDC[d, c, t]) for d in m.D)
            print(f"  t={t:>2}  {c}: Delivered = {delivered:>8.3f}   Demand = {value(m.Demand[c, t]):>8.3f}")
    print()

    # Optional: print arc flows (can be long, but useful for debugging)
    print("PLANT -> DC SHIPMENTS:")
    for t in sorted(m.T):
        for p in m.P:
            for d in m.D:
                x = value(m.ShipPD[p, d, t])
                if x > 1e-6:
                    print(f"  t={t:>2}  {p}->{d}: {x:>8.3f}")
    print()

    print("DC -> CUSTOMER SHIPMENTS:")
    for t in sorted(m.T):
        for d in m.D:
            for c in m.C:
                x = value(m.ShipDC[d, c, t])
                if x > 1e-6:
                    print(f"  t={t:>2}  {d}->{c}: {x:>8.3f}")


# =============================================================================
# 4) MAIN
# =============================================================================
if __name__ == "__main__":
    m = build_model()

    # If you want to DISABLE the quadratic budget constraint:
    # m.QuadBudget.set_value(1e12)

    # Solve (default IPOPT)
    solver_used, _ = solve_model(m, preferred_solver="ipopt")

    # Print solution
    print_solution(m, solver_used)


RuntimeError: No suitable nonlinear/quadratic solver found.
Install IPOPT (free) or use Gurobi/CPLEX.
LP solvers (glpk/cbc/highs) will fail with x^2 terms.